In [1]:
# !pip install datasets
# !pip install transformers[torch]
# !pip install tf-keras

In [2]:
import pandas as pd
from helper_func import clean_text
from helper_func import custom_train_validation_split

# Load data

df_train = pd.read_csv("/home/jack/github/kaggle/scoring/data/train.csv")


df_train.head()


/home/jack/envs/scoring/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-04-23 12:16:44.452595: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-23 12:16:45.047816: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
[nltk_data] Downloading package punkt to /home/jack/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/jack/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       dat

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3


In [3]:
df_train = clean_text(df_train, 'full_text')
df_train.head()

Starting text cleaning process. 



,essay_id,full_text,score,lowered,clean_text
0,000d118,Many people have car where they live. The thin...,3,many people have car where they live. the thin...,many people have car where they live the thing...
1,000fe60,I am a scientist at NASA that is discussing th...,3,i am a scientist at nasa that is discussing th...,i am a scientist at nasa that is discussing th...
2,001ab80,People always wish they had the same technolog...,4,people always wish they had the same technolog...,people always wish they had the same technolog...
3,001bdc0,"We all heard about Venus, the planet without a...",4,"we all heard about venus, the planet without a...",we all heard about venus the planet without al...
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3,"dear, state senator\n\nthis is a letter to arg...",dear state senator this is a letter to argue i...


In [4]:
df_train, df_val = custom_train_validation_split(df_train, 0.3, random_state=42)

In [5]:
df_train.reset_index(drop=True, inplace=True)

In [6]:
df_val.reset_index(drop=True, inplace=True)

In [7]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-large")
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-large")

# Keep the identifier column for reference
train_df = df_train[['essay_id', 'clean_text', 'score']]
val_df = df_val[['essay_id', 'clean_text', 'score']]

# Create DatasetDict with the unique identifier
dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "val": Dataset.from_pandas(val_df),
})

# Tokenize only the text and create a dataset for training
def tokenize_function(examples):
    return tokenizer(
        examples['clean_text'], 
        truncation=True, 
        padding='max_length', 
        max_length=512,
    )

# Tokenize the dataset and keep identifier
tokenized_dataset = dataset_dict.map(
    tokenize_function,
    batched=True,
    remove_columns=['clean_text'],  # Only the tokenized text and other non-text data are kept
)

# Retain the identifier in the dataset for later use
tokenized_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'score', 'essay_id'])

# Define the training arguments
# Adjusted training arguments with logging
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_strategy='steps',  # Log during steps
    logging_steps=10,          # Log every 10 steps
    evaluation_strategy='epoch',  # Evaluate every epoch
    save_strategy='epoch',         # Save every epoch
    load_best_model_at_end=True,  # Load the best model after training
    metric_for_best_model='loss', # Metric used to determine the best model
    report_to='tensorboard',      # Report to TensorBoard
)


from transformers import ProgressCallback, TrainerCallback, TrainerControl, TrainerState, Trainer
import logging

# A custom callback to log additional information
class CustomLoggingCallback(TrainerCallback):
    def on_train_begin(self, args, state, control):
        print("Training is starting")

    def on_train_end(self, args, state, control):
        print("Training is completed")

    def on_step_end(self, args, state, control):
        if state.global_step % 10 == 0:
            print(f"Step {state.global_step}: Loss = {state.log_history[-1]['loss']}")

# Adding the ProgressCallback to track progress with a progress bar
progress_callback = ProgressCallback()

# Recreate the Trainer with additional callbacks
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    callbacks=[progress_callback, CustomLoggingCallback()],
)

# Train the model with callbacks for tracking progress
trainer.train()



Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 5193/5193 [00:01<00:00, 4588.83 examples/s]
You are adding a <class 'transformers.trainer_callback.ProgressCallback'> to the callbacks of this Trainer, but there is already one. The currentlist of callbacks is
:DefaultFlowCallback
TensorBoardCallback
ProgressCallback
CustomLoggingCallback
  0%|          | 0/4545 [00:00<?, ?it/s]

TypeError: CustomLoggingCallback.on_train_begin() got an unexpected keyword argument 'model'

In [ ]:
# Predict and retain the identifier for merging
predictions = trainer.predict(tokenized_dataset['val'])
pred_labels = predictions.predictions.argmax(axis=1)

# Create a DataFrame with predictions and the unique identifier
pred_df = pd.DataFrame({
    'essay_id': val_df['essay_id'],
    'predicted_label': pred_labels,
})

# Now you can merge this prediction DataFrame with other data as needed

In [ ]:
predictions.numpy()